In [1]:
"""
Finite Difference Method (FDM) solver for the 1D complex-valued Helmholtz equation.

Problem:  u''(x) + k² u(x) = f(x),   x ∈ [0, 1]
BCs:      u(0) = 1 + 0i,  u(1) = 0 + 0i

Parameters match the LinPDE-GP experiment exactly.
"""

import numpy as np
from typing import List, Dict, Optional


# ============================================================
# Problem parameters (identical to LinPDE-GP script)
# ============================================================
domain = (0.0, 1.0)

# Physical parameters
rho = 1.0
omega = 2.0
G_real = 2.0
G_imag = 2.0

# Complex wavenumber squared
k_squared = rho * omega**2 / (G_real + 1j * G_imag)
k = np.sqrt(k_squared)

# Source term (constant RHS)
f_val = 2.0 + 3.0j

# Dirichlet boundary conditions
u_left = 1.0 + 0.0j
u_right = 0.0 + 0.0j

# Analytical solution
u_particular = f_val / k_squared
B = 1.0 - u_particular
A = -(B * np.cos(k * 1.0) + u_particular) / np.sin(k * 1.0)


def analytical_solution(x):
    return A * np.sin(k * x) + B * np.cos(k * x) + u_particular


# ============================================================
# Finite Difference Method
# ============================================================
# Use 1000 interior points (matching the evaluation grid density)
N_interior = 998  # so total grid = 1000 points including boundaries
N_total = N_interior + 2
x_fdm = np.linspace(domain[0], domain[1], N_total)
h = x_fdm[1] - x_fdm[0]

# Build the system for interior points: u''(x_i) + k² u(x_i) = f
# Central difference: u''(x_i) ≈ (u_{i-1} - 2u_i + u_{i+1}) / h²
# => (1/h²) u_{i-1} + (-2/h² + k²) u_i + (1/h²) u_{i+1} = f

n = N_interior  # number of unknowns
diag_main = np.full(n, -2.0 / h**2 + k_squared, dtype=complex)
diag_off = np.full(n - 1, 1.0 / h**2, dtype=complex)

A_mat = np.diag(diag_main) + np.diag(diag_off, 1) + np.diag(diag_off, -1)

# RHS vector
rhs = np.full(n, f_val, dtype=complex)

# Apply boundary conditions
rhs[0] -= u_left / h**2
rhs[-1] -= u_right / h**2

# Solve
u_interior = np.linalg.solve(A_mat, rhs)

# Full solution including boundaries
u_fdm = np.empty(N_total, dtype=complex)
u_fdm[0] = u_left
u_fdm[1:-1] = u_interior
u_fdm[-1] = u_right


# ============================================================
# Evaluation (same procedure and format as LinPDE-GP script)
# ============================================================
x_test = np.linspace(0.0, 1.0, 1000)

# Interpolate FDM solution onto the test grid
# (If grids match, this is exact; otherwise use linear interpolation)
pred_complex = np.interp(x_test, x_fdm, np.real(u_fdm)) + \
               1j * np.interp(x_test, x_fdm, np.imag(u_fdm))

pred_re = np.real(pred_complex)
pred_im = np.imag(pred_complex)

# Analytical solution on test grid
u_exact = analytical_solution(x_test)
true_re = np.real(u_exact)
true_im = np.imag(u_exact)


def compute_metrics(u_pred: np.ndarray,
                    u_true: np.ndarray,
                    metrics: Optional[List[str]] = None) -> Dict[str, float]:
    if metrics is None:
        metrics = ["mse", "rmse", "mae", "max_error", "relative_l2", "r2"]

    u_pred = np.atleast_1d(u_pred).flatten()
    u_true = np.atleast_1d(u_true).flatten()

    metric_functions = {
        "mse": lambda yt, yp: np.mean((yt - yp)**2),
        "rmse": lambda yt, yp: np.sqrt(np.mean((yt - yp)**2)),
        "mae": lambda yt, yp: np.mean(np.abs(yt - yp)),
        "max_error": lambda yt, yp: np.max(np.abs(yt - yp)),
        "r2": lambda yt, yp: 1 - np.sum((yt - yp)**2) / np.sum((yt - np.mean(yt))**2),
        "relative_l2": lambda yt, yp: (
            np.linalg.norm(yt - yp) / np.linalg.norm(yt) if np.linalg.norm(yt) > 0 else np.inf
        ),
    }

    return {m: metric_functions[m](u_true, u_pred) for m in metrics if m in metric_functions}


# Per-component metrics
metrics_re = compute_metrics(pred_re, true_re)
metrics_im = compute_metrics(pred_im, true_im)

# Complex magnitude error
complex_error = np.abs(pred_complex - u_exact)
metrics_complex = {
    "mse": np.mean(complex_error**2),
    "rmse": np.sqrt(np.mean(complex_error**2)),
    "mae": np.mean(complex_error),
    "max_error": np.max(complex_error),
    "relative_l2": np.linalg.norm(complex_error) / np.linalg.norm(u_exact),
    "r2": 1 - np.sum(complex_error**2) / np.sum(np.abs(u_exact - np.mean(u_exact))**2),
}


# Print results (same format as LinPDE-GP script)
def _print_block(title, m):
    print(f"\n  --- {title} ---")
    print(f"    R² Score:            {m['r2']:12.6f}")
    print(f"    MSE:                 {m['mse']:12.6e}")
    print(f"    RMSE:                {m['rmse']:12.6e}")
    print(f"    Mean Absolute Error: {m['mae']:12.6e}")
    print(f"    Maximum Error:       {m['max_error']:12.6e}")
    print(f"    Relative L2 Error:   {m['relative_l2']:12.6e}")


print("=" * 58)
print("EVALUATION METRICS — Complex Helmholtz 1D (FDM)".center(58))
print(f"  k² = {k_squared:.4f}  (k = {k:.4f})")
print(f"  Grid points: {N_total}  (h = {h:.6e})")
print("=" * 58)

_print_block("Real Part", metrics_re)
_print_block("Imaginary Part", metrics_im)

print("\n" + "=" * 58)

     EVALUATION METRICS — Complex Helmholtz 1D (FDM)      
  k² = 1.0000-1.0000j  (k = 1.0987-0.4551j)
  Grid points: 1000  (h = 1.001001e-03)

  --- Real Part ---
    R² Score:                1.000000
    MSE:                 2.454308e-15
    RMSE:                4.954097e-08
    Mean Absolute Error: 4.507691e-08
    Maximum Error:       6.833300e-08
    Relative L2 Error:   1.108620e-07

  --- Imaginary Part ---
    R² Score:                1.000000
    MSE:                 6.195400e-17
    RMSE:                7.871086e-09
    Mean Absolute Error: 7.201323e-09
    Maximum Error:       1.075119e-08
    Relative L2 Error:   2.356815e-08



     EVALUATION METRICS — Complex Helmholtz 1D (FDM)      
  k² = 1.0000-1.0000j  (k = 1.0987-0.4551j)
  Grid points: 1000  (h = 1.001001e-03)

  --- Real Part ---
    R² Score:                1.000000
    MSE:                 2.454308e-15
    RMSE:                4.954097e-08
    Mean Absolute Error: 4.507691e-08
    Maximum Error:       6.833300e-08
    Relative L2 Error:   1.108620e-07

  --- Imaginary Part ---
    R² Score:                1.000000
    MSE:                 6.195400e-17
    RMSE:                7.871086e-09
    Mean Absolute Error: 7.201323e-09
    Maximum Error:       1.075119e-08
    Relative L2 Error:   2.356815e-08

